# VSA-Based LISP Interpreter Demo

This notebook demonstrates a LISP interpreter built on Vector Symbolic Architecture (VSA). Every data structure — atoms, cons cells, lists — is represented as a hypervector, and every operation is VSA algebra.

## 1. Setup

Import `LispEnv` and create a fresh in-memory environment. The default model uses 64K-dimensional 8-bit hypervectors.

In [1]:
%pip install -q --upgrade kongming-rs-hv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import uuid

from kongming.lisp import LispEnv

ns = f"user-{uuid.uuid4().hex[:8]}"
env = LispEnv(namespace=ns)
print(f"LispEnv ready, model: {env.model}, namespace: {ns}")

LispEnv ready, model: 1, namespace: user-47130661


## 2. Atoms and Lists

`parse_display` parses an S-expression into hypervectors and prints it back. `QUOTE` returns data unevaluated.

In [3]:
# Parse and display a list (no evaluation)
env.parse_display("(A B C)")

'(A B C)'

In [4]:
# QUOTE returns its argument unevaluated
env.eval("(QUOTE (A B C))")

'(A B C)'

## 3. Primitives: CAR, CDR, CONS

`CAR` returns the first element, `CDR` returns the rest, and `CONS` constructs a pair.

In [5]:
# CAR: first element of a list
env.eval("(CAR (QUOTE (A B C)))")

'A'

In [6]:
# CDR: rest of a list
env.eval("(CDR (QUOTE (A B C)))")

'(B C)'

In [7]:
# CONS: construct a pair
env.eval("(CONS (QUOTE A) (QUOTE B))")

'(A . B)'

## 4. Predicates: ATOM, EQ

`ATOM` tests whether its argument is an atom (not a list). `EQ` tests equality of two atoms.

In [8]:
# ATOM on an atom -> T
env.eval("(ATOM (QUOTE A))")

'T'

In [9]:
# ATOM on a list -> F
env.eval("(ATOM (QUOTE (A B)))")

'F'

In [10]:
# EQ: equal atoms -> T
env.eval("(EQ (QUOTE A) (QUOTE A))")

'T'

In [11]:
# EQ: different atoms -> F
env.eval("(EQ (QUOTE A) (QUOTE B))")

'F'

## 5. Conditional: COND

`COND` evaluates clauses in order and returns the value of the first whose test is true (`T`).

In [12]:
# First clause is false, second (T) is the default branch
env.eval("(COND ((EQ (QUOTE A) (QUOTE B)) (QUOTE NO)) (T (QUOTE YES)))")

'YES'

## 6. Lambda and DEFINE

`LAMBDA` creates anonymous functions. `DEFINE` binds a name to a value in the environment.

In [13]:
# Inline lambda: take the CAR of the argument
env.eval("((LAMBDA (X) (CAR X)) (QUOTE (A B C)))")

'A'

In [14]:
# DEFINE a reusable function SECOND
env.eval("(DEFINE SECOND (LAMBDA (L) (CAR (CDR L))))")

'NIL'

In [15]:
# Call the defined function
env.eval("(SECOND (QUOTE (X Y Z)))")

'Y'

## 7. Recursion with LABEL

`LABEL` enables recursive functions by giving a lambda a self-referencing name. Use `eval_full` for recursive expressions — it evaluates repeatedly until the result stabilizes.

In [16]:
# DEFINE a recursive LAST function using LABEL
env.eval("(DEFINE LAST (LAMBDA (L) ((LABEL REC (LAMBDA (X) (COND ((ATOM (CDR X)) (CAR X)) (T (REC (CDR X)))))) L)))")

'NIL'

In [17]:
# eval_full drives recursive evaluation to completion
env.eval_full("(LAST (QUOTE (A B C)))")

'C'

## 8. Persistent Storage with Fjall

Pass a filesystem `path` to `LispEnv` to back the codebook with fjall, a persistent key-value store. Definitions survive across sessions.

In [18]:
import tempfile

d = tempfile.mkdtemp()
print("Fjall storage path:", d)

fenv = LispEnv(path=d)

Fjall storage path: /var/folders/3c/c1xc7w7j407d7_fg_58x7g600000gn/T/tmpb5t___bv


In [19]:
# Works the same as the in-memory environment
fenv.eval("(CAR (QUOTE (X Y Z)))")

'X'

In [20]:
fenv.eval("(DEFINE GREET (LAMBDA (X) (CONS (QUOTE HELLO) X)))")
fenv.eval("(GREET (QUOTE WORLD))")

'(HELLO . WORLD)'

## 9. Two Interpreters: Rust vs Pure Python

The LISP interpreter ships in two implementations with the same API:

| | `kongming_rs.lisp` | `kongming_rs.pylisp` |
|-|-------------------|---------------------|
| **Engine** | Compiled Rust (via PyO3) | Pure Python |
| **Source** | Inside `kongming-rs-hv` binary | Readable `.py` files ([pylisp/](../pylisp/)) |
| **Use case** | Production, notebooks | Learning, debugging, extending |

Both produce identical results for the same input.

In [21]:
from kongming_rs.lisp import LispEnv as RustLisp
from kongming_rs.pylisp import LispEnv as PyLisp

rust_env = RustLisp()
py_env = PyLisp()

# Same expressions, same results
exprs = [
    ("CAR", "(CAR (QUOTE (A B C)))"),
    ("CDR", "(CDR (QUOTE (A B C)))"),
    ("CONS", "(CONS (QUOTE A) (QUOTE B))"),
    ("ATOM", "(ATOM (QUOTE A))"),
    ("EQ", "(EQ (QUOTE A) (QUOTE A))"),
    ("COND", "(COND ((EQ (QUOTE A) (QUOTE B)) (QUOTE NO)) (T (QUOTE YES)))"),
    ("LAMBDA", "((LAMBDA (X) (CAR X)) (QUOTE (A B C)))"),
]

print(f"{'Expr':<8} {'Rust':<12} {'Python':<12} {'Match'}")
print("-" * 44)
for label, expr in exprs:
    r = rust_env.eval(expr)
    p = py_env.eval(expr)
    print(f"{label:<8} {r:<12} {p:<12} {'✓' if r == p else '✗'}")

Expr     Rust         Python       Match
--------------------------------------------
CAR      A            A            ✓
CDR      (B C)        (B C)        ✓
CONS     (A . B)      (A . B)      ✓
ATOM     T            T            ✓
EQ       T            T            ✓
COND     YES          YES          ✓
LAMBDA   A            A            ✓


In [22]:
# Recursion works identically too
rust_env.eval(
    "(DEFINE LAST (LAMBDA (L) ((LABEL REC (LAMBDA (X) (COND ((ATOM (CDR X)) (CAR X)) (T (REC (CDR X)))))) L)))"
)
py_env.eval("(DEFINE LAST (LAMBDA (L) ((LABEL REC (LAMBDA (X) (COND ((ATOM (CDR X)) (CAR X)) (T (REC (CDR X)))))) L)))")

r = rust_env.eval_full("(LAST (QUOTE (A B C)))")
p = py_env.eval_full("(LAST (QUOTE (A B C)))")
print(f"LAST: Rust={r}, Python={p}, Match={'✓' if r == p else '✗'}")

LAST: Rust=C, Python=C, Match=✓
